In [1]:
%autosave 180
%load_ext autoreload
%autoreload 2

# import matplotlib
# #matplotlib.use('qtagg')
%matplotlib tk
# import matplotlib.pyplot as plt
# plt.ion()
# %matplotlib notebook

#|
import os
import numpy as np
import time
import nidaqmx
from nidaqmx.constants import (AcquisitionType)  # https://nidaqmx-python.readthedocs.io/en/latest/constants.html
import numpy as np
import matplotlib.pyplot as plt
from nidaqmx.constants import TerminalConfiguration
import math
from multiprocessing import Process

# 



Autosaving every 180 seconds


In [22]:
###################################################
###################################################
###################################################
fname_bmi = r'D:\bmi\DON8498\22-06-08\data\Image_001_001.raw'
fname_calibration = r'D:\bmi\DON8498\22-06-08\calibration\Image_001_001.raw'

#
bmi_data = np.memmap(fname_bmi,
                   dtype='uint16',
                   mode='r',
                   shape=20000*512*512).reshape(20000,512,512)

#
calibration_data = np.memmap(fname_calibration,
                   dtype='uint16',
                   mode='r',
                   shape=20000*512*512).reshape(20000,512,512)

#
print (bmi_data.shape, calibration_data.shape)

#
rois = np.load(r'D:\bmi\DON8498\22-06-08\rois_pixels_and_thresholds.npz')

#
contours = []
for k in range(4):
    contours.append(rois['cell'+str(k)+'_contour'])


(20000, 512, 512) (20000, 512, 512)


In [23]:
#################################################
#################################################
#################################################
plt.figure()
ax=plt.subplot(121)
plt.imshow(bmi_data[2000:2500].mean(0))
# add contours on top of cells
for c in range(len(contours)):
    for k in range(len(contours[c])-1):
        plt.plot([contours[c][k][0], contours[c][k+1][0]],
                 [contours[c][k][1], contours[c][k + 1][1]],
                           c='red')
#                
ax=plt.subplot(122)
plt.imshow(calibration_data[2000:2500].mean(0))
for c in range(len(contours)):
    for k in range(len(contours[c])-1):
        plt.plot([contours[c][k][0], contours[c][k+1][0]],
                 [contours[c][k][1], contours[c][k + 1][1]],
                           c='red')

plt.show()

In [7]:
data = np.load('/media/cat/4TB/donato/DON-009460/22-07-20/databmi_results.npz',
               allow_pickle=True)

#
lick_detector_times = data['lick_detector_abstime']
print ("lick detector: ", lick_detector_times.shape, lick_detector_times)

#
reward_times = data['reward_times'].squeeze().T[:,1]
print ("reward times: ", reward_times[:50])
ttl_times = data['ttl_times']
print ("ttl times: ", ttl_times)

#
rot1 = data['rotary_encoder1_abstime']
rot2 = data['rotary_encoder2_abstime']

#




lick detector:  (3009206, 1) [[5.19499007]
 [5.21086134]
 [5.22203601]
 ...
 [5.20195399]
 [5.20567888]
 [5.20924182]]
reward times:  [  435   930  1957  3142  4319  9248  9550 10624 11465 12081 13082 24338
 25180 26391 28298 34520 35206 35973 36970 56341 59727 61185 62203 66343
    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
    -1    -1]
ttl times:  [  28.8398133   28.8747537   28.9068659 ... 3028.7462991 3028.7802078
 3028.8124843]


In [9]:
#######################################################
#######################################################
#######################################################
plt.figure()

plt.plot(np.arange(lick_detector_times.shape[0])/1000., 
         lick_detector_times, 
         label='lick detector')

if False:
    idx = np.where(reward_times>0)[0]
    temp = reward_times[idx]
    print (temp)
    plt.scatter(ttl_times[temp], np.zeros(temp.shape[0])+2, label='reward times')


plt.legend()
#          
plt.show()

/home/cat/anaconda3/envs/bmi/lib/python3.8/tkinter/__init__.py:814: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  func(*args)


In [2]:
#####################################
#####################################
#####################################

#######################################
# FOR WINDOWS
# fname_root_path = r"D:\User Training"
# fname_fluorescence = r"D:\User Training\Readtest1\Image_001_001.raw"
# fname_freq =  r"F:\freq.npy"
# fname_rois = r"D:\User Training\rois.txt"

# FOR LINUX
fname_root_path = '/media/cat/4TB/donato/BSCOPE_tests/'
fname_fluorescence = os.path.join(fname_root_path, 
                                  'image_10000frames.raw')

#
fname_freq =  os.path.join(fname_root_path,
                           "freq.npy")

#
fname_rois = os.path.join(fname_root_path, 
                          "rois.txt")

# required for simulation mode
fname_ttl = os.path.join(fname_root_path,
                         "ttl_pulses.npy")


####################################################################### 			
################### DEFAULT PARAMTERS FOR BMI ######################### 			
####################################################################### 			
sampleRate_2P = 30
n_seconds_session = 10                          # number of seconds to run the BMI 
simulation_mode = True							# Run BMI in simulation mode (i.e. don't need Bscope input)

###############################################################
#################### INITIALIZE BMI ########################### 
###############################################################
bmi = BMI(simulation_mode,
          fname_root_path,
          fname_fluorescence,
          fname_rois,
          fname_freq,
          fname_ttl,
          sampleRate_2P,
          n_seconds_session)

# for simulation mode we sometimes want to slow down the processing;
# ... not as necessary 
bmi.sleep_time_sec = 0.033

# Flag to print out information from the proessing
bmi.verbose = False
bmi.verbose2 = False    # this displays the time it takes to copute ROI


 ROIS: ,  (10, 2)
   using square ROIs; TODO: use proper defined ROIs and cell masks ...
 ttl counter initialized:  [0] psm_f1d4b362
 tone frequency initialized:  [0] psm_244ef8f1


In [3]:
############################################################### 			
#################### INITIALIZE Plotting ###################### 
############################################################### 

#
plotter_ = Process(target=PlotROIs, args=(
                    bmi.shmem_rois_traces.name,
                    bmi.shmem_n_ttl.name,
                    bmi.rois_traces.shape,))

plotter_.start()

#
# plotter_.join()

# 
# plotter_.terminate()

...assuming sampling rate is 
 30 

In [4]:
############################################################### 			
#################### INITIALIZE TONE ########################## 
############################################################### 





In [5]:

#
bmi.run_BMI()


Running BMI (ctrl-c to stop)
TODO: implement shared memory to run plotting and tone playback directly from memroy
  more info:  https://docs.python.org/3/library/multiprocessing.shared_memory.html


% complete:  39%|##########4                | 116/300 [00:00<00:00, 1156.54it/s]

 duration to setup memmap:  0.00014853477478027344  sec.
     TODO: work with 1D flattened arrays


% complete:  89%|########################1  | 268/300 [00:00<00:00, 1366.92it/s]

...Saving BMI metadata...
... DONE BMI...


/home/cat/anaconda3/envs/bmi/lib/python3.8/site-packages/numpy/lib/npyio.py:713: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  val = np.asanyarray(val)
% complete: 100%|##########################9| 299/300 [00:19<00:00, 1366.92it/s]

In [18]:
# NOTES AND SCRATCH SPACE

In [13]:
data = np.load('/media/cat/4TB/donato/BSCOPE_tests/100k_frames_ttl_data.zip')
print (data['data'].shape)

ttl_pulses = data['data']
np.save('/media/cat/4TB/donato/BSCOPE_tests/ttl_pulses.npy', ttl_pulses)

#
plt.figure()
plt.plot(data['data'])
plt.show()

(3599999, 1)


In [23]:
fname_fluorescence = '/media/cat/4TB/donato/BSCOPE_tests/8105/20220309/Image_001_001.raw'
n_frames_to_be_acquired = 1000
data_Ca = np.memmap(fname_fluorescence, dtype='uint16', mode='r', 
									   shape=(n_frames_to_be_acquired,512,512))
print (data_Ca.shape)
    
data_Ca.tofile('/media/cat/4TB/donato/BSCOPE_tests/image_1000frames.raw')

(1000, 512, 512)


In [2]:
data = np.load('/home/cat/rois_traces.npy')
print (data.shape)

(10, 600)


In [5]:
for k in range(data.shape[0]):
    plt.plot(data[k]+k*1000)
    
plt.show()